<a href="https://colab.research.google.com/github/mistyy12/flyrank-ML/blob/main/notebooks_finished/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mistyy12/flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
%pip -q install duckdb huggingface_hub

In [1]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The unit of analysis here is the content page, where each row represents one unique content_hash_id for a client on a particular dateand contains the features used to evaluate further action in the page. The time window that I'll use is between January 2026 to April 2026 for training data.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Start with January to test the checks on one partition.
import pandas as pd
results = []

for month in pd.period_range("2026-01", "2026-04", freq="M"):
  source = (
      f"read_parquet('{REL}/"
      f"fact_content_daily_performance/month={month}/*.parquet')"
  )
  result = con.sql(f"""
    SELECT
      COUNT(DISTINCT content_hash_id) AS unique_pages,
      COUNT(DISTINCT client_hash_id) AS unique_clients,
      COUNT(*) AS total_rows,
      MIN(report_date) AS first_date,
      MAX(report_date) AS last_date
    FROM {source}
  """).df()

  #Python: list.insert(position, column_name, value)
  result.insert(0, "month", str(month))
  results.append(result)

monthly_summary = pd.concat(results, ignore_index=True)
display(monthly_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,unique_pages,unique_clients,total_rows,first_date,last_date
0,2026-01,261984,42,7890817,2026-01-01,2026-01-31
1,2026-02,321546,54,7355108,2026-02-01,2026-02-28
2,2026-03,331437,55,9841378,2026-03-01,2026-03-31
3,2026-04,362172,61,10424730,2026-04-01,2026-04-30


In [12]:
# Example 2 -> January 2025 source
jan2025_source = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2025-01/*.parquet')"
)

display(con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {jan2025_source}
""").df())

,total_rows,unique_clients,unique_pages,first_date,last_date
0,1297,2,476,2025-01-27,2025-01-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Daily performance fields
display(con.sql(f"""
    DESCRIBE SELECT * FROM {source}
""").df())

# Content metadata fields
display(con.sql(f"""
    DESCRIBE SELECT * FROM {TABLES['dim_content']}
""").df())

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


1. Feature ->
    "content_created_date","content_updated_date", "gsc_impresssions", "gsc_avg_position", "gsc_sum_position",
    "gsc_clicks",
    "gsc_impressions".
2. Context
-> "content_hash_id", "client_hash_id", "report_date", "month"  

3. Label inputs
-> Outcome-window impressions used to construct something like "is_declining_label"

4. Excluded
-> Outcome-window measurements from model inputs because they reveal the outcome or unused field.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the partitions
paths = [
    f"{REL}/fact_content_daily_performance/month={month}/*.parquet"
    for month in pd.period_range("2026-01", "2026-04", freq="M")
]

# A view defines the source without loading everything into pandas.
con.read_parquet(paths).create_view("contract_daily", replace=True)


┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude │ ai_

In [22]:
# Count -> to check total_rows, unique clients, unique pages in each rows.
# Windows -> first_date, last_date, dates_present (date/time window boundary)
# Grain: missing keys -> how many rows lack at least one field to identify a client-page-date record.
# Measurement availability: how many rows lack confirmed available measurements, even if metrics contain zeros.
# Missing values: how many rows have NULL measurements

window_checks = con.sql("""
    SELECT
        CASE
            WHEN report_date < DATE '2026-04-01'
                THEN 'feature_window'
            ELSE 'outcome_window'
        END AS window_role,

        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_pages,


        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS dates_present,



        COUNT(*) FILTER (
            WHERE client_hash_id IS NULL
               OR content_hash_id IS NULL
               OR report_date IS NULL
        ) AS missing_grain_fields,


        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS missing_impressions,

        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS missing_clicks,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        ) AS missing_position,


        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS gsc_unavailable_or_unknown,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT TRUE
        ) AS ga4_unavailable_or_unknown

    FROM contract_daily
    GROUP BY window_role
    ORDER BY first_date
""").df()

display(window_checks)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,window_role,total_rows,unique_clients,unique_pages,first_date,last_date,dates_present,missing_grain_fields,missing_impressions,missing_clicks,missing_position,gsc_unavailable_or_unknown,ga4_unavailable_or_unknown
0,feature_window,25087303,59,349411,2026-01-01,2026-03-31,90,0,98006,98006,16457317,16457316,24412230
1,outcome_window,10424730,61,362172,2026-04-01,2026-04-30,30,0,0,0,6523681,6523670,9900439


In [17]:
#Verify the daily grain. One record per client-page-date in these 4 partitions

display(con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM contract_daily
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Overall, this features includes dates used to calculate the outcome (April). Therefore, my Jan-Mar feature window and April outcome window do not overlap.

However, there are some clients that have 120 observed dates, while others have fewer, such as clients with 1 or 11 dates.

Besides that, GSC is available on dates when GA4 is unavailable (GSC-only periods). This unavailable GA4 values cannot be interpreted as zero engagement.

This may cause unbalanced history and overlapping windows, as the data is also taken from last year.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
with pd.option_context("display.max_rows", None):
  display(con.sql("""
      SELECT
          client_hash_id,
          MIN(report_date) AS first_date,
          MAX(report_date) AS last_date,
          COUNT(DISTINCT report_date) AS dates_present,
          COUNT(DISTINCT report_date) FILTER (
              WHERE gsc_data_available IS TRUE
          ) AS dates_with_gsc,
          COUNT(DISTINCT report_date) FILTER (
              WHERE ga4_data_available IS TRUE
          ) AS dates_with_ga4
      FROM contract_daily
      GROUP BY client_hash_id
      ORDER BY dates_present, dates_with_ga4
  """).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,first_date,last_date,dates_present,dates_with_gsc,dates_with_ga4
0,client_c353557474475e51,2026-04-30,2026-04-30,1,1,0
1,client_7de9989c909e91a5,2026-04-20,2026-04-30,11,0,6
2,client_9c26c096d6e57253,2026-04-16,2026-04-30,15,2,4
3,client_0b245132bb722950,2026-04-06,2026-04-30,25,19,7
4,client_8ddc46da5414ffd8,2026-04-02,2026-04-30,29,24,0
5,client_06d356715a8ff3b6,2026-04-02,2026-04-30,29,21,25
6,client_e00b29e582949543,2026-03-23,2026-04-30,39,35,37
7,client_810019792c9b8efc,2026-03-20,2026-04-30,42,37,33
8,client_f6f0cdf26d03d7bd,2026-03-19,2026-04-30,43,43,6
9,client_861cdcccf8049915,2026-01-01,2026-02-23,54,52,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.